In [ ]:
"""
Support Vector Regression (SVR)
Flexural Strength Prediction of Gypsum-Based Composites

Author: Haseeb AHmad
Year: 2026
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


# ==========================================================
# DATA LOADING & PREPROCESSING
# ==========================================================
def load_data(filepath):
    """Load dataset and filter rows with valid flexural strength."""
    
    df = pd.read_csv(filepath, header=2)
    df_array = np.asarray(df)

    # Input features
    X_raw = np.column_stack([
        df_array[0:161, 4],
        df_array[0:161, 5],
        df_array[0:161, 6],
        df_array[0:161, 7],
        df_array[0:161, 8],
        df_array[0:161, 9],
        df_array[0:161, 10]
    ]).astype(float)

    # Target (Flexural Strength)
    y_raw = df_array[0:161, 12].astype(float)

    # Remove NaN rows
    mask = ~np.isnan(y_raw)
    X = X_raw[mask]
    y = y_raw[mask]

    return X, y


# ==========================================================
# MODEL TRAINING
# ==========================================================
def train_svm(X_train, y_train):
    """Train SVR model."""
    model = SVR(kernel="rbf", C=1.0, epsilon=0.1)
    model.fit(X_train, y_train)
    return model


# ==========================================================
# EVALUATION
# ==========================================================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2


def print_metrics(train_metrics, test_metrics):
    print("=" * 60)
    print("MODEL PERFORMANCE METRICS")
    print("=" * 60)

    print("\nTRAINING SET:")
    print(f"MSE  : {train_metrics[0]:.6f}")
    print(f"RMSE : {train_metrics[1]:.6f}")
    print(f"MAE  : {train_metrics[2]:.6f}")
    print(f"R²   : {train_metrics[3]:.6f}")

    print("\nTESTING SET:")
    print(f"MSE  : {test_metrics[0]:.6f}")
    print(f"RMSE : {test_metrics[1]:.6f}")
    print(f"MAE  : {test_metrics[2]:.6f}")
    print(f"R²   : {test_metrics[3]:.6f}")
    print("=" * 60)


# ==========================================================
# VISUALIZATIONS
# ==========================================================
def plot_learning_curve(model, X, y, save_path=None):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y,
        scoring="r2",
        train_sizes=np.linspace(0.3, 1.0, 20),
        shuffle=True,
        random_state=RANDOM_STATE
    )

    plt.figure(figsize=(5, 4))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training R²")
    plt.plot(train_sizes, test_scores.mean(axis=1), marker="s", label="Testing R²")
    plt.xlabel("Training Set Size")
    plt.ylabel("R² Score")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()


def plot_actual_vs_predicted(y_true, y_pred, save_path=None):
    x_vals = np.linspace(y_true.min(), y_true.max(), 200)

    plt.figure(figsize=(5, 4))
    plt.scatter(y_true, y_pred, alpha=0.6, edgecolors='k')
    plt.plot(x_vals, x_vals, 'r--', label="Perfect Fit")
    plt.fill_between(x_vals, 0.9*x_vals, 1.1*x_vals, alpha=0.2, label="±10% Error")
    plt.fill_between(x_vals, 0.8*x_vals, 1.2*x_vals, alpha=0.1, label="±20% Error")

    plt.xlabel("Actual Values (MPa)")
    plt.ylabel("Predicted Values (MPa)")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()


def plot_feature_importance(model, X_test, y_test, feature_names, save_path=None):
    result = permutation_importance(
        model, X_test, y_test,
        n_repeats=20,
        scoring="neg_mean_squared_error",
        random_state=RANDOM_STATE
    )

    importances = np.abs(result.importances_mean)
    indices = np.argsort(importances)

    plt.figure(figsize=(5, 4))
    plt.barh(range(len(importances)), importances[indices])
    plt.yticks(range(len(importances)), np.array(feature_names)[indices])
    plt.xlabel("Permutation Importance")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()


# ==========================================================
# MAIN PIPELINE
# ==========================================================
def main():

    data_path = os.path.join("..", "data", "Gypsum_updated.csv")

    X, y = load_data(data_path)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    print("Training SVM model...")
    model = train_svm(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_metrics = evaluate(y_train, y_train_pred)
    test_metrics = evaluate(y_test, y_test_pred)

    print_metrics(train_metrics, test_metrics)

    feature_names = [
        "Gypsum Strength", "Gypsum Quantity", "Water Quantity",
        "Water/Gypsum Ratio", "Wheat Straw", "CaCl2", "Ca(OH)2"
    ]

    plot_learning_curve(model, X_train, y_train,
                        save_path="../figures/learning_curve.png")

    plot_actual_vs_predicted(y_test, y_test_pred,
                             save_path="../figures/actual_vs_predicted.png")

    plot_feature_importance(model, X_test, y_test,
                            feature_names,
                            save_path="../figures/feature_importance.png")


if __name__ == "__main__":
    main()